# Merged Model Eval - 2026-06-11

Analyze CARC eval results exported from `eval_merged_*_to_*_11_06_2026_*` runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
CSV_PATH = Path("eval_results.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("scripts/plotting/eval_merged_11_06_2026/eval_results.csv")
CSV_PATH = CSV_PATH.resolve()

SPLIT = "test"
METRIC = "roc_auc"
METRICS = ["accuracy", "f1", "roc_auc"]

SHOT_ORDER = [0, 3, 10]
DATASET_ORDER = [
    "covid19_twitter",
    "ukr_rus_twitter",
    "midterm",
    "covid_political",
    "election2020",
    "ukr_rus_suspended",
]
TASK_ORDER = ["nm", "lp", "pl"]
TASK_LABELS = {
    "nm": "Neighbor matching",
    "lp": "Temporal link prediction",
    "pl": "Classification",
}
MODEL_ORDER = ["merged_nm", "merged_lp"]
MODEL_LABELS = {
    "merged_nm": "Merged NM",
    "merged_lp": "Merged LP",
}

In [ ]:
df = pd.read_csv(CSV_PATH)

required = {"model", "dataset", "task", "shots", "split", *METRICS}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

df["shots"] = df["shots"].astype(int)
for col in METRICS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

unexpected_models = sorted(set(df["model"]) - set(MODEL_ORDER))
unexpected_shots = sorted(set(df["shots"]) - set(SHOT_ORDER))
unexpected_splits = sorted(set(df["split"]) - {SPLIT})
if unexpected_models:
    raise ValueError(f"Unexpected model values: {unexpected_models}")
if unexpected_shots:
    raise ValueError(f"Unexpected shot values: {unexpected_shots}")
if unexpected_splits:
    raise ValueError(f"Unexpected split values: {unexpected_splits}")

df["dataset"] = pd.Categorical(df["dataset"], categories=DATASET_ORDER, ordered=True)
df["task"] = pd.Categorical(df["task"], categories=TASK_ORDER, ordered=True)
df["model"] = pd.Categorical(df["model"], categories=MODEL_ORDER, ordered=True)
df["model_label"] = df["model"].astype(str).map(MODEL_LABELS)
df["task_label"] = df["task"].astype(str).map(TASK_LABELS)
df = df.sort_values(["split", "dataset", "task", "model", "shots"])

print(f"reading {CSV_PATH}")
print(f"rows: {len(df)}")
display(df.head())

In [ ]:
coverage = (
    df.groupby(["model", "dataset", "task"], observed=True)
      .agg(shots=("shots", lambda x: sorted(set(x))), rows=("shots", "size"))
      .reset_index()
      .sort_values(["model", "dataset", "task"])
)

display(coverage)

wide_coverage = (
    df.groupby(["dataset", "task", "shots", "model"], observed=True)
      .size()
      .rename("rows")
      .reset_index()
      .pivot_table(
          index=["dataset", "task", "shots"],
          columns="model",
          values="rows",
          fill_value=0,
          observed=True,
      )
      .reset_index()
      .sort_values(["dataset", "task", "shots"])
)

display(wide_coverage)

In [ ]:
def plot_metric_grid(data, *, split=SPLIT, metric=METRIC, datasets=DATASET_ORDER, tasks=TASK_ORDER):
    subset = data[(data["split"] == split) & data[metric].notna()].copy()
    if subset.empty:
        raise ValueError(f"No rows for split={split!r}, metric={metric!r}")

    palette = dict(zip(MODEL_ORDER, sns.color_palette("Set2", n_colors=len(MODEL_ORDER))))
    fig, axes = plt.subplots(
        nrows=len(datasets),
        ncols=len(tasks),
        figsize=(4.7 * len(tasks), 2.6 * len(datasets)),
        sharex=True,
        sharey=True,
    )

    if len(datasets) == 1 and len(tasks) == 1:
        axes = [[axes]]
    elif len(datasets) == 1:
        axes = [axes]
    elif len(tasks) == 1:
        axes = [[ax] for ax in axes]

    for row_idx, dataset in enumerate(datasets):
        for col_idx, task in enumerate(tasks):
            ax = axes[row_idx][col_idx]
            panel = subset[(subset["dataset"].astype(str) == dataset) & (subset["task"].astype(str) == task)]
            ax.set_title(f"{dataset}\n{TASK_LABELS.get(task, task)}", fontsize=10)

            if panel.empty:
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center", color="0.45")
            else:
                for model in MODEL_ORDER:
                    line = panel[panel["model"].astype(str) == model].sort_values("shots")
                    if line.empty:
                        continue
                    ax.plot(
                        line["shots"],
                        line[metric],
                        marker="o",
                        linewidth=2,
                        markersize=5,
                        color=palette[model],
                        label=MODEL_LABELS.get(model, model),
                    )

            ax.set_ylim(0, 1.02)
            ax.set_xticks(SHOT_ORDER)
            ax.grid(True, alpha=0.3)
            if row_idx == len(datasets) - 1:
                ax.set_xlabel("shots")
            if col_idx == 0:
                ax.set_ylabel(metric)

    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, title="model", loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.01))
    fig.suptitle(f"Merged eval results: {split} {metric}", y=1.035, fontsize=14)
    fig.tight_layout()
    return fig

plot_metric_grid(df, metric=METRIC)
plt.show()

In [ ]:
for metric in METRICS:
    plot_metric_grid(df, metric=metric)
    plt.show()

In [ ]:
delta = (
    df.pivot_table(
        index=["dataset", "task", "shots"],
        columns="model",
        values=METRICS,
        observed=True,
    )
    .reset_index()
)

for metric in METRICS:
    delta[(metric, "merged_nm_minus_lp")] = delta[(metric, "merged_nm")] - delta[(metric, "merged_lp")]

delta.columns = ["_".join([str(part) for part in col if str(part)]) if isinstance(col, tuple) else col for col in delta.columns]
display(delta.sort_values(["dataset", "task", "shots"]))

In [ ]:
delta_metric = f"{METRIC}_merged_nm_minus_lp"
heat = delta.pivot_table(
    index=["dataset", "task"],
    columns="shots",
    values=delta_metric,
    observed=True,
)

fig, ax = plt.subplots(figsize=(7.5, 7.0))
sns.heatmap(heat, annot=True, fmt=".3f", center=0, cmap="vlag", linewidths=0.5, ax=ax)
ax.set_title(f"Delta {METRIC}: Merged NM - Merged LP")
ax.set_xlabel("shots")
ax.set_ylabel("dataset / task")
fig.tight_layout()
plt.show()

In [ ]:
best_rows = []
for metric in METRICS:
    idx = df.groupby(["model", "dataset", "task"], observed=True)[metric].idxmax()
    best = df.loc[idx, ["model_label", "dataset", "task", "shots", metric]].copy()
    best["metric"] = metric
    best = best.rename(columns={metric: "value"})
    best_rows.append(best)

best_table = pd.concat(best_rows, ignore_index=True)
display(best_table[["metric", "model_label", "dataset", "task", "shots", "value"]].sort_values(["metric", "dataset", "task", "model_label"]))

In [ ]:
# Optional: save the main figure locally next to the CSV.
FIGURE_DIR = CSV_PATH.parent / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

for metric in METRICS:
    fig = plot_metric_grid(df, metric=metric)
    fig.savefig(FIGURE_DIR / f"eval_merged_11_06_2026_{metric}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print(f"wrote figures to {FIGURE_DIR}")